In [31]:
import requests
import pandas as pd

BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"

def fetch_eurostat(dataset, params):
    url = BASE_URL + dataset
    response = requests.get(url, params=params)

    if response.status_code != 200:
        raise Exception(f"Eurostat API error {response.status_code}: {response.text}")

    return response.json()


In [ ]:
#Eurostat API parameters
params = {
    "format": "json",
    "lang": "en",
    "geo": "DE",           # Germany
    "nace_r2": "G47",      # Retail trade, except motor vehicles/motorcycles (all sub-categories)
    "s_adj": "CA",         # Calendar adjusted, not seasonally adjusted
    "indic_bt": "NETTUR",  # Net turnover 
    "freq": "M",           # Monthly
    "unit": "I21"          # Index, base year 2021 = 100
}



In [33]:
data = fetch_eurostat("sts_trtu_m", params)
data


{'version': '2.0',
 'class': 'dataset',
 'label': 'Turnover and volume of sales in wholesale and retail trade - monthly data',
 'source': 'ESTAT',
 'updated': '2026-08-01T11:00:00+0200',
 'value': {'36': 64.8,
  '37': 62.7,
  '38': 69.8,
  '39': 68.0,
  '40': 66.1,
  '41': 62.6,
  '42': 64.1,
  '43': 63.3,
  '44': 66.3,
  '45': 68.9,
  '46': 71.1,
  '47': 86.2,
  '48': 64.8,
  '49': 64.3,
  '50': 70.7,
  '51': 69.1,
  '52': 68.9,
  '53': 64.8,
  '54': 65.8,
  '55': 64.4,
  '56': 66.6,
  '57': 68.8,
  '58': 72.2,
  '59': 84.9,
  '60': 65.6,
  '61': 63.2,
  '62': 69.7,
  '63': 70.5,
  '64': 68.0,
  '65': 63.6,
  '66': 66.5,
  '67': 64.3,
  '68': 67.2,
  '69': 69.6,
  '70': 72.1,
  '71': 84.0,
  '72': 66.7,
  '73': 62.9,
  '74': 69.4,
  '75': 69.3,
  '76': 67.4,
  '77': 64.1,
  '78': 65.7,
  '79': 61.9,
  '80': 65.4,
  '81': 70.2,
  '82': 71.0,
  '83': 83.5,
  '84': 65.5,
  '85': 62.7,
  '86': 70.9,
  '87': 69.2,
  '88': 68.2,
  '89': 63.0,
  '90': 66.9,
  '91': 63.5,
  '92': 65.9,
  '93'

In [34]:

data = fetch_eurostat("sts_trtu_m", params)
print("Number of value entries:", len(data['value']))

Number of value entries: 389


In [ ]:
def jsonstat_to_df(data):
    time_index = data['dimension']['time']['category']['index']
    pos_to_time = {v: k for k, v in time_index.items()}
    
    records = []
    for pos_str, val in data['value'].items():
        pos = int(pos_str)
        records.append({'date': pos_to_time[pos], 'value': val})
    
    df = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
    df['date'] = pd.to_datetime(df['date'])
    
    for dim in ['geo', 'nace_r2', 's_adj', 'unit', 'indic_bt']:
        code = list(data['dimension'][dim]['category']['index'].keys())[0]
        df[dim] = code
    
    return df


In [36]:
df = jsonstat_to_df(data)
df.head()

,date,value,geo,nace_r2,s_adj,unit,indic_bt
0,1994-01-01,64.8,DE,G47,CA,I21,NETTUR
1,1994-02-01,62.7,DE,G47,CA,I21,NETTUR
2,1994-03-01,69.8,DE,G47,CA,I21,NETTUR
3,1994-04-01,68.0,DE,G47,CA,I21,NETTUR
4,1994-05-01,66.1,DE,G47,CA,I21,NETTUR


In [37]:
print(df.shape)
df.tail()

(389, 7)


,date,value,geo,nace_r2,s_adj,unit,indic_bt
384,2026-01-01,111.2,DE,G47,CA,I21,NETTUR
385,2026-02-01,107.8,DE,G47,CA,I21,NETTUR
386,2026-03-01,125.2,DE,G47,CA,I21,NETTUR
387,2026-04-01,121.8,DE,G47,CA,I21,NETTUR
388,2026-05-01,123.6,DE,G47,CA,I21,NETTUR


In [38]:
import os
from datetime import datetime

def load_incremental(new_df, csv_path):
    new_df = new_df.copy()
    new_df['loaded_at'] = datetime.now().isoformat()
    
    if not os.path.exists(csv_path):
        new_df.to_csv(csv_path, index=False)
        print(f"Initial load: {len(new_df)} rows")
        return
    
    existing_df = pd.read_csv(csv_path, parse_dates=['date'])
    key_cols = ['date', 'geo', 'nace_r2', 's_adj', 'unit', 'indic_bt']
    
    merged = new_df.merge(
        existing_df[key_cols + ['value']],
        on=key_cols, suffixes=('_new', '_old'), how='left'
    )
    
    new_rows = merged[merged['value_old'].isna()]
    changed_rows = merged[(merged['value_old'].notna()) & (merged['value_new'] != merged['value_old'])]
    print(f"New months: {len(new_rows)}, Revised months: {len(changed_rows)}")
    
    unchanged_keys = existing_df.merge(changed_rows[key_cols], on=key_cols, how='left', indicator=True)
    unchanged = unchanged_keys[unchanged_keys['_merge'] == 'left_only'].drop(columns=['_merge'])
    
    to_update_keys = pd.concat([new_rows, changed_rows])[key_cols].apply(tuple, axis=1)
    to_write = new_df[new_df[key_cols].apply(tuple, axis=1).isin(to_update_keys)]
    
    final_df = pd.concat([unchanged, to_write], ignore_index=True)
    final_df.to_csv(csv_path, index=False)
    print(f"Saved: {len(final_df)} total rows")

In [39]:
print(os.getcwd())

c:\Users\HP 250 G7\GitRepos\Personal_Projects\sql-data-warehouse-original\sql-data-warehouse-project\api_integration


In [ ]:
# create the folder and store CSV
output_dir = r"C:\Users\HP 250 G7\GitRepos\Personal_Projects\sql-data-warehouse-original\sql-data-warehouse-project\datasets\source_eurostat"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "retail_trade_germany_monthly.csv")
load_incremental(df, csv_path)

Initial load: 389 rows


In [41]:
check = pd.read_csv(csv_path)
print(check.shape)
check.tail()

(389, 8)


,date,value,geo,nace_r2,s_adj,unit,indic_bt,loaded_at
384,2026-01-01,111.2,DE,G47,CA,I21,NETTUR,2026-08-02T19:52:14.408588
385,2026-02-01,107.8,DE,G47,CA,I21,NETTUR,2026-08-02T19:52:14.408588
386,2026-03-01,125.2,DE,G47,CA,I21,NETTUR,2026-08-02T19:52:14.408588
387,2026-04-01,121.8,DE,G47,CA,I21,NETTUR,2026-08-02T19:52:14.408588
388,2026-05-01,123.6,DE,G47,CA,I21,NETTUR,2026-08-02T19:52:14.408588
